In [3]:
import jax
import jax.numpy as jnp
from jax.numpy.fft import rfftn, irfftn
import numpy as np

import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

import importlib
import model.model
import model.grid
import model
importlib.reload(model.model)
importlib.reload(model.utils)
importlib.reload(model.grid)

from model.model import QGM
from model.utils import Solver 
from model.grid import Grid

jax.config.update("jax_platform_name", "cpu")

# === Parameter definitions === #
master_key = jax.random.PRNGKey(0)
master_key, key_init, init1, init2 = jax.random.split(master_key, num=4)

params = {
        'nx': 256, 
        'Lx': 2*jnp.pi, 
        'beta': 0, 
        'k_beta':6, #for now we're just doing this for the forcing range idgaf
        'eta': 0, 
        'mu': 0, # linear drag
        'nu': 0, # hyperviscosity
        'dt': 5e-3,
        'k_f': 8.0,        # central forcing wavenumber
        'k_width': 1.5,    # width of the ring
        'epsilon': 1e-4,   #energy input 
        'key':key_init
}

# === Set up model === 
model = QGM(params) 
model.initialize()

def step_diagnostic(model):
 z0 = jnp.asarray(model.fields['zeta'])
 qh0 = rfftn(z0, axes=(-2, -1))
 KE0 = 0.5 * jnp.sum(jnp.abs(qh0)**2 * model.grid.invK2) / (model.grid.Lx * model.grid.Ly)
 # take one step
 model.step()
 z1 = jnp.asarray(model.fields['zeta'])
 qh1 = rfftn(z1, axes=(-2, -1))
 KE1 = 0.5 * jnp.sum(jnp.abs(qh1)**2 * model.grid.invK2) / (model.grid.Lx * model.grid.Ly)
 diff_L2 = jnp.linalg.norm((z1 - z0).ravel())
 diff_max = jnp.max(jnp.abs(z1 - z0))
 print(f"KE before={float(KE0):.6e}, after={float(KE1):.6e}, KE change={(float(KE1)-float(KE0)):.6e}")
 print(f"L2 diff={float(diff_L2):.6e}, max diff={float(diff_max):.6e}")
 return dict(KE0=float(KE0), KE1=float(KE1), L2=float(diff_L2), max=float(diff_max))


# Re-initialize model, then run a short trace printing per-step KE and L2-change
model.initialize()
nsteps = 200
print('step, KE, KE_change, L2_diff, max_diff')
z_prev = jnp.asarray(model.fields['zeta'])
qh_prev = rfftn(z_prev, axes=(-2,-1))
KE_prev = float(0.5 * jnp.sum(jnp.abs(qh_prev)**2 * model.grid.invK2) / (model.grid.Lx * model.grid.Ly))
for i in range(1, nsteps+1):
 model.step()
 z = jnp.asarray(model.fields['zeta'])
 qh = rfftn(z, axes=(-2,-1))
 KE = float(0.5 * jnp.sum(jnp.abs(qh)**2 * model.grid.invK2) / (model.grid.Lx * model.grid.Ly))
 diff = z - z_prev
 L2 = float(jnp.linalg.norm(diff.ravel()))
 m = float(jnp.max(jnp.abs(diff)))
 print(f"{i:4d}, {KE:.6e}, {(KE-KE_prev):+.6e}, {L2:.6e}, {m:.6e}")
 # update prev
 z_prev = z
 KE_prev = KE
 # quick break if changes are effectively zero
 if L2 < 1e-12 and abs(KE-KE_prev) < 1e-14:
   print('Evolution numerically negligible; stopping early at step', i)
   break

step, KE, KE_change, L2_diff, max_diff
   1, 4.999999e+03, -9.765625e-04, 4.002841e-03, 1.288280e-04
   2, 4.999996e+03, -2.441406e-03, 4.002921e-03, 1.288801e-04
   3, 4.999996e+03, +0.000000e+00, 4.002930e-03, 1.288578e-04
   4, 4.999997e+03, +4.882812e-04, 4.002913e-03, 1.288652e-04
   5, 4.999995e+03, -1.953125e-03, 4.003072e-03, 1.288764e-04
   6, 4.999993e+03, -1.464844e-03, 4.002993e-03, 1.288988e-04
   7, 4.999993e+03, +0.000000e+00, 4.003000e-03, 1.289099e-04
   8, 4.999991e+03, -2.441406e-03, 4.002923e-03, 1.288913e-04
   9, 4.999991e+03, +0.000000e+00, 4.003040e-03, 1.289099e-04
  10, 4.999989e+03, -1.464844e-03, 4.003029e-03, 1.289248e-04
  11, 4.999988e+03, -9.765625e-04, 4.003065e-03, 1.288839e-04
  12, 4.999988e+03, +0.000000e+00, 4.002996e-03, 1.289397e-04
  13, 4.999988e+03, -4.882812e-04, 4.003109e-03, 1.289248e-04
  14, 4.999987e+03, -9.765625e-04, 4.003156e-03, 1.289472e-04
  15, 4.999986e+03, -9.765625e-04, 4.003166e-03, 1.289323e-04
  16, 4.999985e+03, -4.882812e-

In [4]:
import jax, jax.numpy as jnp
from jax.numpy.fft import rfftn, irfftn
import numpy as np

def compute_urms(model):
    """Return U_rms and V_rms (floats) computed from fields['psi'] if present, else compute from zeta."""
    grid = model.grid
    # ensure psi field exists
    psi = jnp.asarray(model.fields.get("psi", None))
    if psi is None or psi.size == 0:
        # compute psi from zeta (spectral Poisson)
        qh = rfftn(jnp.asarray(model.fields["zeta"]), axes=(-2, -1))
        psih = - qh * grid.invK2
        psi = irfftn(psih, axes=(-2, -1)).real
    u =  jnp.gradient(psi, grid.dy, axis=-2)   # dψ/dy
    v = -jnp.gradient(psi, grid.dx, axis=-1)   # -dψ/dx
    U_rms = float(jnp.sqrt(jnp.mean(u**2 + v**2)))
    return U_rms, float(jnp.sqrt(jnp.mean(u**2))), float(jnp.sqrt(jnp.mean(v**2)))

def rhs_components_diag(model):
    """Non-jitted diagnostic that computes norms/maxes of RHS pieces (spectral)."""
    grid = model.grid
    params = model.parameters
    zeta = jnp.asarray(model.fields["zeta"])
    qh = rfftn(zeta, axes=(-2,-1))
    psih = - qh * grid.invK2
    uh = -1j * grid.KY * psih
    vh =  1j * grid.KX * psih
    u, v, q = irfftn(jnp.stack([uh, vh, qh], axis=0), axes=(-2,-1)).real

    # nonlinear
    qe = q + params.get("eta", 0.0)
    uqh, vqh = rfftn(jnp.stack([u*qe, v*qe], axis=0), axes=(-2,-1))
    nonlinear_h = -1j * (grid.KX * uqh + grid.KY * vqh)

    beta = params.get("beta", 0.0)
    beta_h = - beta * 1j * grid.KX * psih

    # forcing (deterministic sample for diagnostic)
    key = getattr(model, "_key", model.parameters.get("key", jax.random.PRNGKey(0)))
    key_d, _ = jax.random.split(key)
    noise_phys = jax.random.normal(key_d, (grid.ny, grid.nx))
    forcing_h = rfftn(noise_phys, axes=(-2,-1)) * model.forcing_spectrum

    nu = params.get("nu", 0.0)
    m = params.get("m", 4)
    mu = params.get("mu", 0.0)
    diss_h = mu * qh + nu * (grid.Kmag**m) * qh

    rhs_h = nonlinear_h + beta_h - diss_h + forcing_h

    def L2(a): return float(jnp.linalg.norm(a.ravel()))
    def maxabs(a): return float(jnp.max(jnp.abs(a)))

    out = {
      "KE": float(0.5 * jnp.sum(jnp.abs(qh)**2 * grid.invK2) / (grid.Lx * grid.Ly)),
      "U_rms_estimate": compute_urms(model)[0],
      "||nonlinear_h||_2": L2(nonlinear_h),
      "max|nonlinear_h|": maxabs(nonlinear_h),
      "||beta_h||_2": L2(beta_h),
      "max|beta_h|": maxabs(beta_h),
      "||forcing_h||_2": L2(forcing_h),
      "max|forcing_h|": maxabs(forcing_h),
      "||diss_h||_2": L2(diss_h),
      "max|diss_h|": maxabs(diss_h),
      "||rhs_h||_2": L2(rhs_h),
      "max|rhs_h|": maxabs(rhs_h),
    }

    for k,v in out.items():
        print(f"{k:20s}: {v:.6e}")
    return out

In [5]:
def estimate_advective_change_per_step(model):
    """Rough estimate: advective change ~ dt * <|k|> * U_rms * q_rms."""
    grid = model.grid
    zeta = jnp.asarray(model.fields["zeta"])
    qh = rfftn(zeta, axes=(-2,-1))
    # q_rms (physical)
    q_rms = float(jnp.sqrt(jnp.mean(zeta**2)))
    # typical wavenumber weighted by power
    power = jnp.abs(qh)**2
    Kmag = grid.Kmag
    weighted = jnp.sum(power * Kmag) / (jnp.sum(power) + 1e-16)
    ktyp = float(weighted)
    U_rms, _, _ = compute_urms(model)
    dt = model.parameters.get("dt", 1e-3)
    advect_scale = dt * ktyp * U_rms * q_rms
    print(f"ktyp={ktyp:.3e}, U_rms={U_rms:.3e}, q_rms={q_rms:.3e}, dt={dt:.3e}")
    print(f"Estimated advective change magnitude per step ~ {advect_scale:.6e}")
    return advect_scale

In [6]:
# 3a) Put energy at larger scales (lower k) and raise E0
def set_band_initial(model, E0=1.0, kmin=2, kmax=6, seed=1234):
    grid = model.grid
    L = grid.Lx
    kmin_val = kmin * 2*jnp.pi / L
    kmax_val = kmax * 2*jnp.pi / L
    key = jax.random.PRNGKey(seed)
    noise = jax.random.normal(key, (grid.ny, grid.nx))
    qh = rfftn(noise, axes=(-2,-1))
    band_mask = (grid.Kmag >= kmin_val) & (grid.Kmag <= kmax_val)
    qh = qh * band_mask
    qh = qh.at[:, 0].set(0.0)
    E_spec = 0.5 * jnp.sum(jnp.abs(qh)**2 * grid.invK2) / (grid.Lx * grid.Ly)
    scale = jnp.sqrt(E0 / (E_spec + 1e-16))
    qh = qh * scale
    zeta0 = irfftn(qh, axes=(-2,-1)).real
    model._initial = jax.device_put(zeta0)
    model.initialize()
    print("Initial KE:", float(0.5 * jnp.sum(jnp.abs(rfftn(model.fields['zeta'], axes=(-2,-1)))**2 * model.grid.invK2) / (model.grid.Lx * model.grid.Ly)))

# 3b) Increase dt (careful: may destabilize)
def set_dt_and_reinit(model, dt):
    model.parameters['dt'] = dt
    model.initialize()
    print("dt set to", dt)

# 3c) Increase forcing amplitude (epsilon) and/or move forcing to lower k
def amplify_forcing(model, factor=10.0, k_f=None):
    model.parameters['epsilon'] = model.parameters.get('epsilon', 0.0) * factor
    if k_f is not None:
        model.parameters['k_f'] = k_f
    # recompute forcing_spectrum if needed (simplest: re-create model or update forcing_spectrum manually)
    grid = model.grid
    kf = model.parameters.get('k_f', 8.0)
    kw = model.parameters.get('k_width', 1.5)
    fs = jnp.exp(- (grid.Kmag - kf)**2 / (2 * kw**2))
    fs = jnp.where(grid.K2 == 0, 0.0, fs)
    eps0 = jnp.sum(fs * grid.invK2 / 2) / (grid.Lx * grid.Ly)
    fs = fs * (model.parameters['epsilon'] / (eps0 + 1e-20))
    model.forcing_spectrum = fs
    print("epsilon now", model.parameters['epsilon'])

In [7]:
rhs_components_diag(model)
estimate_advective_change_per_step(model)
set_band_initial(model, E0=50000.0, kmin=2, kmax=6)

KE                  : 5.000011e+03
U_rms_estimate      : 1.351699e-02
||nonlinear_h||_2   : 1.472490e+02
max|nonlinear_h|    : 2.441457e+01
||beta_h||_2        : 0.000000e+00
max|beta_h|         : 0.000000e+00
||forcing_h||_2     : 1.069279e+01
max|forcing_h|      : 2.982273e+00
||diss_h||_2        : 0.000000e+00
max|diss_h|         : 0.000000e+00
||rhs_h||_2         : 1.475385e+02
max|rhs_h|          : 2.336379e+01
ktyp=7.171e+00, U_rms=1.352e-02, q_rms=8.819e-02, dt=5.000e-03
Estimated advective change magnitude per step ~ 4.274444e-05
Initial KE: 50000.0


In [8]:
import jax
import jax.numpy as jnp
from jax.numpy.fft import rfftn, irfftn

def set_initial_by_psi_target_U(model, target_U_rms=0.1, kmin=1, kmax=3, seed=1234, max_scale=1e3):
    """
    Build a band-limited psi_hat (spectral), scale its amplitude so that resulting U_rms ~= target_U_rms,
    convert to vorticity qh = -K^2 * psi_hat, and set model._initial accordingly.
    This avoids dividing by invK2 and gives direct control of velocities.
    """
    grid = model.grid
    L = grid.Lx

    # spectral band in absolute wavenumber units (integer multiples)
    kmin_val = kmin * 2*jnp.pi / L
    kmax_val = kmax * 2*jnp.pi / L

    # random complex spectral psi (create real-space noise then FFT for Hermitian symmetry)
    key = jax.random.PRNGKey(seed)
    noise = jax.random.normal(key, (grid.ny, grid.nx))
    psi_h = rfftn(noise, axes=(-2, -1))

    # band-limit in K magnitude
    mask = (grid.Kmag >= kmin_val) & (grid.Kmag <= kmax_val)
    psi_h = psi_h * mask

    # zero mean (optional)
    psi_h = psi_h.at[0, 0].set(0.0)

    # compute velocities from this psi_h to get current U_rms
    uh = -1j * grid.KY * psi_h
    vh =  1j * grid.KX * psi_h
    u, v = irfftn(jnp.stack([uh, vh], axis=0), axes=(-2, -1)).real
    U_rms_current = float(jnp.sqrt(jnp.mean(u**2 + v**2)) + 1e-30)

    if U_rms_current == 0:
        raise RuntimeError("constructed psi_h has zero velocity (mask too narrow?). Try a wider band or different seed.")

    # scale factor to reach target_U_rms (clamped)
    scale = float(target_U_rms / U_rms_current)
    scale = max(min(scale, max_scale), 1.0 / max_scale)

    psi_h *= scale

    # now compute qh consistent with psi_h
    qh = - (grid.K2) * psi_h
    # ensure real-valued physical zonal mean is real
    qh = qh.at[:, 0].set(jnp.real(qh[:, 0]))

    # set as initial zeta and initialize
    zeta0 = irfftn(qh, axes=(-2, -1)).real
    model._initial = jax.device_put(zeta0)
    model.initialize()

    # report
    uh2 = -1j * grid.KY * rfftn(model.fields['psi'], axes=(-2,-1))
    vh2 =  1j * grid.KX * rfftn(model.fields['psi'], axes=(-2,-1))
    u_now, v_now = irfftn(jnp.stack([uh2, vh2], axis=0), axes=(-2, -1)).real
    U_rms_after = float(jnp.sqrt(jnp.mean(u_now**2 + v_now**2)))
    print(f"Target U_rms={target_U_rms:.3e}, achieved U_rms={U_rms_after:.3e}, scale applied={scale:.3e}")
    return U_rms_after

In [9]:
set_initial_by_psi_target_U(model)



Target U_rms=1.000e-01, achieved U_rms=9.295e-02, scale applied=1.829e+00


0.09295357763767242

In [10]:
model.make_animation(nsteps=2500, frame_interval=100, outname="outputs/test.gif")